# ShiftNotes Prototype

This notebook demonstrates the prototype flow:

Mock Shift Notes -> Cleaning -> Trend Analysis -> Insight Generation -> Weekly Operations Summary

## Step 1 — Imports
Standard libraries for data manipulation and pattern matching.
`signal_classifier.py` handles all signal detection logic and is imported separately in Step 4.

In [1]:
import pandas as pd
import re
from collections import Counter

pd.set_option('display.max_colwidth', 120)

## Step 2 — Load Data
Load two files:
- `mock_shift_notes.csv` — 100 synthetic shift reports across 6 kiosks over 5 weeks
- `ground_truth_targets.csv` — 7 known patterns planted in the dataset to validate pipeline output

In [2]:
notes_path = 'mock_shift_notes.csv'
truth_path = 'ground_truth_targets.csv'

df = pd.read_csv(notes_path)
truth = pd.read_csv(truth_path)

print(f'Reports loaded: {len(df)}')
display(df.head(3))
display(truth)

Reports loaded: 100


,report_id,date,week,kiosk,lead_name,food_quality_rating,food_quantity_rating,food_concerns_or_outages,team_members_who_did_well,guest_issues_for_the_day,operational_notes,number_of_unclaimed_lunches
0,1,2026-05-01,1,Kiosk A,M.T.,4,4,Chicken ran low during the lunch rush — had to stretch remaining portions.,Team stayed focused throughout the shift.,Multiple guests asked about poke — seems like a recurring request.,No operational concerns today.,5
1,2,2026-05-02,1,Kiosk A,J.S.,4,3,Chicken ran low around noon. Stretched what we had.,Solid effort across the board.,Had a few guests ask about poke today.,NaN,5
2,3,2026-05-03,1,Kiosk A,L.P.,4,4,Chicken ran low around the lunch rush; we stretched portions and got through it.,Everyone pulled their weight today.,No guest concerns today.,Nothing to report.,3


,metric,expected_value,note
0,total_reports,100,Total mock records
1,poke_request_mentions,18,Guest request trend
2,chicken_shortage_mentions,12,Food shortage trend
3,highest_waste_kiosk,Kiosk B,Based on unclaimed lunches
4,high_recognition_kiosk,Kiosk E,Frequent recognition language
5,ops_friction_kiosk,Kiosk D,Recurring operational issues
6,inventory_inconsistency_kiosk,Kiosk F,Lowest avg food quantity rating


## Step 3 — Data Cleaning
Normalize all freeform text columns: fill nulls, strip whitespace, and standardize types.

All four text fields are combined into a single `full_text` column, which serves as the primary input to the hybrid signal classifier.

In [3]:
text_cols = [
    'food_concerns_or_outages',
    'team_members_who_did_well',
    'guest_issues_for_the_day',
    'operational_notes'
]

for col in text_cols:
    df[col] = df[col].fillna('').astype(str).str.strip()

df['date'] = pd.to_datetime(df['date'])
df['week'] = df['week'].astype(int)

df['full_text'] = (
    df['food_concerns_or_outages'] + ' ' +
    df['team_members_who_did_well'] + ' ' +
    df['guest_issues_for_the_day'] + ' ' +
    df['operational_notes']
).str.lower()

display(df[['report_id', 'date', 'kiosk', 'week']].head())

,report_id,date,kiosk,week
0,1,2026-05-01,Kiosk A,1
1,2,2026-05-02,Kiosk A,1
2,3,2026-05-03,Kiosk A,1
3,4,2026-05-04,Kiosk A,1
4,5,2026-05-02,Kiosk B,1


## Step 4 — Signal Detection (Hybrid Classifier)
Detect four operational signals from each shift note using a two-stage hybrid approach:

1. **Regex fast-path** — exact pattern matching, instant and free
2. **HuggingFace zero-shot fallback** — resolves ambiguous language that regex misses

Each row receives four boolean signal flags plus an audit trail showing which signals came from regex vs HuggingFace.

| Signal | Example |
|---|---|
| `chicken_shortage` | "ran out of chicken" |
| `poke_request` | "guests asked about poke" |
| `ops_issue` | "register froze during lunch" |
| `team_recognition` | "shoutout to Dom B." |

In [4]:
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

from signal_classifier import apply_to_dataframe

df = apply_to_dataframe(df, text_col='full_text')

display(df[['report_id', 'kiosk',
            'chicken_shortage_mention', 'poke_request_mention',
            'ops_issue_mention', 'team_recognition_mention',
            'regex_hits', 'hf_hits']].head(10))

INFO: Classifying 100 rows from column 'full_text'...


INFO: Loading HuggingFace model (first run may take a moment)...


INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/nli-MiniLM2-L6-H768/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/nli-MiniLM2-L6-H768/b95119ce93d3e065de6214e38cd4a97b0f2f2c6d/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

INFO:   chicken_shortage_mention: 12 hits


INFO:   poke_request_mention: 18 hits


INFO:   ops_issue_mention: 17 hits


INFO:   team_recognition_mention: 31 hits


,report_id,kiosk,chicken_shortage_mention,poke_request_mention,ops_issue_mention,team_recognition_mention,regex_hits,hf_hits
0,1,Kiosk A,True,True,False,False,"[chicken_shortage, poke_request]",[]
1,2,Kiosk A,True,True,False,False,"[chicken_shortage, poke_request]",[]
2,3,Kiosk A,True,False,False,False,[chicken_shortage],[]
3,4,Kiosk A,True,False,False,False,[chicken_shortage],[]
4,5,Kiosk B,False,False,False,False,[],[]
5,6,Kiosk B,False,False,False,True,[],[team_recognition]
6,7,Kiosk B,False,False,False,True,[],[team_recognition]
7,8,Kiosk B,False,False,False,False,[],[]
8,9,Kiosk C,False,True,False,False,[poke_request],[]
9,10,Kiosk C,False,False,False,False,[],[]


## Step 5 — Kiosk Summary
Aggregate signal counts and performance metrics by kiosk.

This table reveals which kiosks have recurring issues, high waste, strong team performance, or inventory inconsistencies.

In [5]:
kiosk_summary = df.groupby('kiosk').agg(
    reports=('report_id', 'count'),
    avg_food_quality=('food_quality_rating', 'mean'),
    avg_food_quantity=('food_quantity_rating', 'mean'),
    total_unclaimed=('number_of_unclaimed_lunches', 'sum'),
    poke_mentions=('poke_request_mention', 'sum'),
    chicken_shortages=('chicken_shortage_mention', 'sum'),
    ops_issues=('ops_issue_mention', 'sum'),
    recognition_mentions=('team_recognition_mention', 'sum')
).reset_index()

kiosk_summary = kiosk_summary.sort_values('kiosk')
display(kiosk_summary)

,kiosk,reports,avg_food_quality,avg_food_quantity,total_unclaimed,poke_mentions,chicken_shortages,ops_issues,recognition_mentions
0,Kiosk A,17,4.000000,3.529412,57,5,12,1,3
1,Kiosk B,17,4.000000,4.000000,140,1,0,0,4
2,Kiosk C,17,4.000000,4.000000,61,7,0,0,3
3,Kiosk D,17,3.764706,4.000000,58,3,0,16,4
4,Kiosk E,16,4.750000,4.000000,16,0,0,0,16
5,Kiosk F,16,4.000000,3.500000,68,2,0,0,1


## Step 6 — Weekly Trend Summary
Aggregate signal counts and unclaimed lunch totals by week.

This view reveals how operational patterns shift over time.

In [6]:
weekly_summary = df.groupby('week').agg(
    reports=('report_id', 'count'),
    poke_mentions=('poke_request_mention', 'sum'),
    chicken_shortages=('chicken_shortage_mention', 'sum'),
    unclaimed_lunches=('number_of_unclaimed_lunches', 'sum'),
    ops_issues=('ops_issue_mention', 'sum')
).reset_index().sort_values('week')

display(weekly_summary)

,week,reports,poke_mentions,chicken_shortages,unclaimed_lunches,ops_issues
0,1,21,3,4,79,3
1,2,24,4,3,77,5
2,3,24,2,2,123,4
3,4,28,9,3,106,4
4,5,3,0,0,15,1


## Step 7 — Ground Truth Validation
Compare extracted signal totals against the 7 known patterns planted in the mock dataset.

All rows should show `PASS` if the pipeline is working correctly.

In [7]:
# Validate extracted totals against planted ground truth
extracted = {
    'total_reports': int(len(df)),
    'poke_request_mentions': int(df['poke_request_mention'].sum()),
    'chicken_shortage_mentions': int(df['chicken_shortage_mention'].sum()),
    'highest_waste_kiosk': kiosk_summary.sort_values('total_unclaimed', ascending=False).iloc[0]['kiosk'],
    'high_recognition_kiosk': kiosk_summary.sort_values('recognition_mentions', ascending=False).iloc[0]['kiosk'],
    'ops_friction_kiosk': kiosk_summary.sort_values('ops_issues', ascending=False).iloc[0]['kiosk'],
    'inventory_inconsistency_kiosk': kiosk_summary.sort_values('avg_food_quantity').iloc[0]['kiosk']
}

check_rows = []
for _, row in truth.iterrows():
    metric = row['metric']
    expected = str(row['expected_value'])
    actual = str(extracted.get(metric, 'N/A'))
    status = 'PASS' if actual == expected else 'REVIEW'
    check_rows.append({'metric': metric, 'expected': expected, 'actual': actual, 'status': status})

checks = pd.DataFrame(check_rows)
display(checks)

,metric,expected,actual,status
0,total_reports,100,100,PASS
1,poke_request_mentions,18,18,PASS
2,chicken_shortage_mentions,12,12,PASS
3,highest_waste_kiosk,Kiosk B,Kiosk B,PASS
4,high_recognition_kiosk,Kiosk E,Kiosk E,PASS
5,ops_friction_kiosk,Kiosk D,Kiosk D,PASS
6,inventory_inconsistency_kiosk,Kiosk F,Kiosk F,PASS


## Step 8 — Weekly Operational Briefings
Generate a plain-text operational briefing for each week.

This is the final output — the format intended for delivery to operations leadership.

In [8]:
def weekly_briefing(week_num):
    wk = df[df['week'] == week_num]
    if wk.empty:
        return f'Week {week_num}: no data.'

    waste_by_kiosk = wk.groupby('kiosk')['number_of_unclaimed_lunches'].sum().sort_values(ascending=False)
    top_waste = waste_by_kiosk.index[0]

    lines = [
        f'Week {week_num} Operations Summary',
        f'- Reports analyzed: {len(wk)}',
        f'- Poke requests: {int(wk["poke_request_mention"].sum())}',
        f'- Chicken shortages: {int(wk["chicken_shortage_mention"].sum())}',
        f'- Operational issue mentions: {int(wk["ops_issue_mention"].sum())}',
        f'- Total unclaimed lunches: {int(wk["number_of_unclaimed_lunches"].sum())}',
        f'- Highest weekly waste kiosk: {top_waste}',
        'Recommended attention:',
        f'1. Review prep planning at {top_waste} to reduce unclaimed lunches.',
        '2. Track recurring menu requests to evaluate demand opportunities.',
        '3. Review recurring shortage and ops-friction notes for root causes.'
    ]
    return '\n'.join(lines)

for w in sorted(df['week'].unique()):
    print(weekly_briefing(int(w)))
    print('-' * 60)

Week 1 Operations Summary
- Reports analyzed: 21
- Poke requests: 3
- Chicken shortages: 4
- Operational issue mentions: 3
- Total unclaimed lunches: 79
- Highest weekly waste kiosk: Kiosk B
Recommended attention:
1. Review prep planning at Kiosk B to reduce unclaimed lunches.
2. Track recurring menu requests to evaluate demand opportunities.
3. Review recurring shortage and ops-friction notes for root causes.
------------------------------------------------------------
Week 2 Operations Summary
- Reports analyzed: 24
- Poke requests: 4
- Chicken shortages: 3
- Operational issue mentions: 5
- Total unclaimed lunches: 77
- Highest weekly waste kiosk: Kiosk B
Recommended attention:
1. Review prep planning at Kiosk B to reduce unclaimed lunches.
2. Track recurring menu requests to evaluate demand opportunities.
3. Review recurring shortage and ops-friction notes for root causes.
------------------------------------------------------------
Week 3 Operations Summary
- Reports analyzed: 24
-